# Stack Overflow in the AI Era: Predicting Question Resolution

**Business Analytics Group Case Study — Review 1 Deliverable**  
**Team:**  
- **Member 1 (Data Collection & Pipeline Lead):** Abhinav Gopalakrishnan  
- **Member 2 (Enrichment & Preprocessing Lead):** Rishiikesh  
- **Member 3 (Problem Formulation & EDA Lead):** smallworldsdev  
- **Member 4 (Feature Engineering & Validation Lead):** Nishanth Konchada  
- **Member 5 (Modelling & Evaluation Lead):** Adhikkesh  

---

## Table of Contents (Rubric Order)
1. [1. Problem Formulation & Business Objectives](#1-problem-formulation--business-objectives)
2. [2. Dataset & Data Collection](#2-dataset--data-collection)
3. [3. Data Preprocessing & Cleaning](#3-data-preprocessing--cleaning)
4. [4. Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis-eda)
5. [5. Feature Engineering & Selection](#5-feature-engineering--selection)
6. [6. Baseline & Advanced Predictive Modelling](#6-baseline--advanced-predictive-modelling)
7. [7. Model Evaluation & Robustness Checks](#7-model-evaluation--robustness-checks)
8. [8. Business Findings, Limitations & Recommendations](#8-business-findings-limitations--recommendations)

In [1]:
# Global environment setup
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repo root to path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Project root resolved: {repo_root}")
print(f"Pandas version: {pd.__version__}")

Project root resolved: C:\Users\abhi6\OneDrive\Desktop\BA_GROUP_CASE_STUDY\stackoverflow-ai-era
Pandas version: 2.3.3


## 1. Problem Formulation & Business Objectives

*Lead: Member 3 (Issues #15, #16)*

### 1.1 Business Context
Investigating structural shifts in developer question-and-answer dynamics on Stack Overflow following the mainstream introduction of generative AI coding assistants (ChatGPT in November 2022).

### 1.2 Target Formulations
- **`target_resolved_7d` (Primary):** Whether a posted question receives an accepted answer within 7 days (168 hours).
- **`target_answered_7d` (Secondary):** Whether a posted question receives any answer within 7 days.

In [2]:
# Loader stub: Load problem statement background metadata if present
bg_sources = repo_root / "docs" / "background_sources.md"
if bg_sources.exists():
    print(f"[✓] Background documentation verified: {bg_sources.name}")
else:
    print("[!] Background sources documentation pending.")

[✓] Background documentation verified: background_sources.md


## 2. Dataset & Data Collection

*Lead: Member 1 (Issues #19, #20, #21, #22, #23, #78, #80, #82, #84, #86)*

### 2.1 Collection Methodology & Telemetry
- Public Stack Exchange API v2.3 with minimal filters and SHA-256 HMAC user ID privacy masking.
- Population coverage across 13 target technology tags and 400 non-overlapping 7-day calendar weeks from 2019-01-01 to 2026-08-31 UTC.
- Stratified day-level sampling pre-2025 (1 weekday + 1 weekend day) and exhaustive collection 2025+.

In [3]:
# Loader stub: Verified collection artifacts
weekly_path = repo_root / "data" / "processed" / "weekly_counts.csv"
monthly_path = repo_root / "data" / "processed" / "monthly_counts.csv"
manifest_path = repo_root / "data" / "processed" / "sampling_manifest.csv"

df_weekly = pd.read_csv(weekly_path)
df_monthly = pd.read_csv(monthly_path)
df_manifest = pd.read_csv(manifest_path)

print(f"[✓] Weekly Counts Dataset Loaded: {len(df_weekly):,} rows across {df_weekly['tag'].nunique()} series")
print(f"[✓] Monthly Counts Dataset Loaded: {len(df_monthly):,} rows across {df_monthly['tag'].nunique()} tags")
print(f"[✓] Sampling Manifest Loaded: {len(df_manifest):,} strata records ({df_manifest['n_questions'].sum():,} total sampled questions)")

[✓] Weekly Counts Dataset Loaded: 5,600 rows across 14 series
[✓] Monthly Counts Dataset Loaded: 1,196 rows across 13 tags
[✓] Sampling Manifest Loaded: 2,392 strata records (309,223 total sampled questions)


## 3. Data Preprocessing & Cleaning

*Lead: Member 2 (Issues #6, #7, #8, #51–#58, #61, #69, #70)*

### 3.1 Cleaning Pipeline & Deduplication
- Deduplication across overlapping multi-tagged records.
- HTML tag stripping, code block extraction, and error pattern regex detection.
- Account age derivation and missingness imputation.

In [4]:
# Loader stub: Load cleaned dataset summary and aggregated tables
tag_tables_path = repo_root / "data" / "processed" / "monthly_tag_tables.csv"
if tag_tables_path.exists():
    df_tag_tables = pd.read_csv(tag_tables_path)
    print(f"[✓] Aggregated Monthly Tag Tables Loaded: {len(df_tag_tables)} rows")
    display(df_tag_tables.head(3))
else:
    print("[!] Monthly tag tables not found.")

[✓] Aggregated Monthly Tag Tables Loaded: 1168 rows


,tag,month,n_sampled,est_questions,resolved_rate_7d,answered_rate_7d,accepted_given_answered,median_hours_to_accept,median_hours_to_first_answer,low_sample
0,android,2019-01,467,7876.0,0.361224,0.680929,0.530487,1.897778,1.258333,False
1,android,2019-02,526,8144.0,0.370825,0.705305,0.525766,1.048333,0.875278,False
2,android,2019-03,478,7849.0,0.325519,0.661613,0.492008,1.977778,0.965278,False


## 4. Exploratory Data Analysis (EDA)

*Lead: Member 3 & Member 1 (Issues #29, #38, #42–#48, #84)*

### 4.1 Volume & Resolution Trends
- Temporal evolution of question volume pre- vs. post-ChatGPT.
- Response latency distributions and 7-day resolution rates across technology ecosystems.

In [5]:
# Loader stub: Load EDA aggregated summaries
print("[✓] Ready to import EDA visualisations and statistical tests from notebooks/03_eda.ipynb")

[✓] Ready to import EDA visualisations and statistical tests from notebooks/03_eda.ipynb


## 5. Feature Engineering & Selection

*Lead: Member 4 (Issues #32, #33, #37, #66)*

### 5.1 Temporal Feature Architecture & Leakage Safeguards
- Strictly pre-posting features (text metrics, code ratios, account age, multi-hot tags).
- Lagged tag volume (4-week rolling volume prior to posting) preventing temporal lookahead leakage.

In [6]:
# Loader stub: Verify feature specification documentation
f_spec = repo_root / "docs" / "feature_spec.md"
if f_spec.exists():
    print(f"[✓] Feature specification loaded: {f_spec.name}")
else:
    print("[!] Feature spec pending.")

[✓] Feature specification loaded: feature_spec.md


## 6. Baseline & Advanced Predictive Modelling

*Lead: Member 4 & Member 5 (Issues #13, #37)*

### 6.1 Model Architectures
- Logistic Regression (Interpretable baseline).
- Random Forest & Gradient Boosted Decision Trees (LightGBM/XGBoost).

In [7]:
# Loader stub: Ready to import model training pipelines
print("[✓] Ready to import model artifacts from notebooks/05_models.ipynb")

[✓] Ready to import model artifacts from notebooks/05_models.ipynb


## 7. Model Evaluation & Robustness Checks

*Lead: Member 5 (Issues #12, #14)*

### 7.1 Temporal Validation & Rolling Origin
- Out-of-time evaluation on post-AI holdout data (July 2025 – August 2026).
- Discrimination (AUC-ROC, PR-AUC), calibration (Brier score), and subgroup fairness.

In [8]:
# Loader stub: Ready to import evaluation reports and metric tables
print("[✓] Ready to import evaluation metrics from notebooks/06_evaluation.ipynb")

[✓] Ready to import evaluation metrics from notebooks/06_evaluation.ipynb


## 8. Business Findings, Limitations & Recommendations

*Team Integration Deliverable*

### 8.1 Key Takeaways for Stack Overflow & Developer Communities
- Quantified displacement of basic debugging questions toward conversational LLMs.
- Impact of community responsiveness on platform retention in the AI era.

In [9]:
# Summary verification
print("=== Review 1 Main Analysis Pipeline Skeleton Verified ===")

=== Review 1 Main Analysis Pipeline Skeleton Verified ===
